In [18]:
CATALOG = "movie_analytics"
RAW_SCHEMA = "raw"
BRONZE_SCHEMA = "bronze"
RAW_VOLUME = "source_files"

RAW_ROOT = f"/Volumes/{CATALOG}/{RAW_SCHEMA}/{RAW_VOLUME}"
BRONZE_NAMESPACE = f"{CATALOG}.{BRONZE_SCHEMA}"

In [19]:
files = [
    "name.basics",
    "title.basics",
    "title.crew",
    "title.principals",
    "title.ratings",
]

for file in files:
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("sep", "\t")
        .csv(f"{RAW_ROOT}/imdb/{file}.tsv.gz")
    )

    display(df.limit(5))

,nconst,primaryName,birthYear,deathYear,primaryProfession,knownForTitles
0,nm0000001,Fred Astaire,1899,1987,"actor,miscellaneous,producer","tt0072308,tt0031983,tt0050419,tt0025164"
1,nm0000002,Lauren Bacall,1924,2014,"actress,miscellaneous,soundtrack","tt0037382,tt0117057,tt0075213,tt0038355"
2,nm0000003,Brigitte Bardot,1934,2025,"actress,music_department,producer","tt0057345,tt0049189,tt0056404,tt0054452"
3,nm0000004,John Belushi,1949,1982,"actor,writer,music_department","tt0072562,tt0077975,tt0080455,tt0078723"
4,nm0000005,Ingmar Bergman,1918,2007,"writer,director,actor","tt0050986,tt0083922,tt0069467,tt0050976"


,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894,\N,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892,\N,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892,\N,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892,\N,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893,\N,1,Short


,tconst,directors,writers
0,tt0000001,nm0005690,\N
1,tt0000002,nm0721526,\N
2,tt0000003,nm0721526,nm0721526
3,tt0000004,nm0721526,\N
4,tt0000005,nm0005690,\N


,tconst,ordering,nconst,category,job,characters
0,tt0000001,1,nm1588970,self,\N,"[""Self""]"
1,tt0000001,2,nm0005690,director,\N,\N
2,tt0000001,3,nm0005690,producer,producer,\N
3,tt0000001,4,nm0374658,cinematographer,director of photography,\N
4,tt0000002,1,nm0721526,director,\N,\N


,tconst,averageRating,numVotes
0,tt0000001,5.7,2231
1,tt0000002,5.4,325
2,tt0000003,6.4,2385
3,tt0000004,5.0,202
4,tt0000005,6.2,3096


In [20]:
files = ["links", "movies", "ratings", "tags"]

for file in files:
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{RAW_ROOT}/movielens/{file}.csv")
    )

    display(df.limit(5))

,movieId,imdbId,tmdbId
0,1,114709,862
1,2,113497,8844
2,3,113228,15602
3,4,114885,31357
4,5,113041,11862


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


,userId,movieId,tag,timestamp
0,22,26479,Kevin Kline,1583038886
1,22,79592,misogyny,1581476297
2,22,247150,acrophobia,1622483469
3,34,2174,music,1249808064
4,34,2174,weird,1249808102


# Bronze

## Shared

In [21]:
from datetime import date
from pyspark.sql.functions import current_timestamp, lit

SNAPSHOT_DATE = date(2026, 9, 20)

#### MovieLens

In [22]:
base_path = f"{RAW_ROOT}/movielens"

files = ["links", "movies", "ratings", "tags"]

for name in files:
    path = f"{base_path}/{name}.csv"

    df = (
        spark.read
        .option("header", True)
        .csv(path)
        .withColumn("_source_file", lit(path))
        .withColumn("_source_dataset", lit("movielens_32m"))
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_snapshot_date", lit(SNAPSHOT_DATE))
    )

    target_table = f"{BRONZE_NAMESPACE}.movielens_{name}"
    snapshot_predicate = (
        f"_source_snapshot_date = '{SNAPSHOT_DATE.isoformat()}'"
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("replaceWhere", snapshot_predicate)
        .saveAsTable(target_table)
    )

#### IMDb

In [23]:
base_path = f"{RAW_ROOT}/imdb"

files = [
    "name.basics",
    "title.basics",
    "title.crew",
    "title.principals",
    "title.ratings",
]

for name in files:
    path = f"{base_path}/{name}.tsv.gz"

    df = (
        spark.read
        .option("header", True)
        .option("sep", "\t")
        .csv(path)
        .withColumn("_source_file", lit(path))
        .withColumn("_source_dataset", lit("imdb"))
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_snapshot_date", lit(SNAPSHOT_DATE))
    )

    table_name = name.replace(".", "_")
    target_table = f"{BRONZE_NAMESPACE}.imdb_{table_name}"
    snapshot_predicate = (
        f"_source_snapshot_date = '{SNAPSHOT_DATE.isoformat()}'"
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("replaceWhere", snapshot_predicate)
        .saveAsTable(target_table)
    )

#### Data check

In [24]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
)

imdb_bronze_tables = [
    f"{BRONZE_NAMESPACE}.imdb_name_basics",
    f"{BRONZE_NAMESPACE}.imdb_title_basics",
    f"{BRONZE_NAMESPACE}.imdb_title_crew",
    f"{BRONZE_NAMESPACE}.imdb_title_principals",
    f"{BRONZE_NAMESPACE}.imdb_title_ratings",
]

results = []

for table_name in imdb_bronze_tables:
    df = spark.table(table_name)

    # Exclude Bronze ingestion metadata.
    source_columns = [
        column
        for column in df.columns
        if not column.startswith("_")
    ]

    # Calculate the row count and all \N counts in one scan.
    aggregations = [
        F.count(F.lit(1)).alias("_total_rows")
    ]

    aggregations.extend([
        F.sum(
            F.when(
                F.col(column).cast("string") == r"\N",
                1,
            ).otherwise(0)
        )
        .cast("long")
        .alias(column)
        for column in source_columns
    ])

    # An aggregate always produces one result row.
    counts = df.agg(*aggregations).collect()[0].asDict()

    total_rows = counts["_total_rows"]

    for column in source_columns:
        marker_count = counts[column] or 0

        marker_percentage = (
            round((marker_count / total_rows) * 100, 4)
            if total_rows > 0
            else 0.0
        )

        results.append((
            table_name,
            column,
            total_rows,
            marker_count,
            marker_percentage,
        ))

result_schema = StructType([
    StructField("table_name", StringType(), False),
    StructField("column_name", StringType(), False),
    StructField("total_rows", LongType(), False),
    StructField("backslash_n_count", LongType(), False),
    StructField("backslash_n_percentage", DoubleType(), False),
])

bronze_null_marker_profile = spark.createDataFrame(
    results,
    schema=result_schema,
)

# Show every checked column.
display(
    bronze_null_marker_profile.orderBy(
        "table_name",
        "column_name",
    )
)

# Show only columns containing literal \N values.
display(
    bronze_null_marker_profile
    .filter(F.col("backslash_n_count") > 0)
    .orderBy(
        F.desc("backslash_n_count"),
        "table_name",
        "column_name",
    )
)

,table_name,column_name,total_rows,backslash_n_count,backslash_n_percentage
0,movie_analytics.bronze.imdb_name_basics,birthYear,15668421,14986500,95.6478
1,movie_analytics.bronze.imdb_name_basics,deathYear,15668421,15405054,98.3191
2,movie_analytics.bronze.imdb_name_basics,knownForTitles,15668421,1893385,12.0841
3,movie_analytics.bronze.imdb_name_basics,nconst,15668421,0,0.0000
4,movie_analytics.bronze.imdb_name_basics,primaryName,15668421,89,0.0006
5,movie_analytics.bronze.imdb_name_basics,primaryProfession,15668421,3177194,20.2777
6,movie_analytics.bronze.imdb_title_basics,endYear,12801419,12639581,98.7358
7,movie_analytics.bronze.imdb_title_basics,genres,12801419,542734,4.2396
8,movie_analytics.bronze.imdb_title_basics,isAdult,12801419,0,0.0000
9,movie_analytics.bronze.imdb_title_basics,originalTitle,12801419,0,0.0000


,table_name,column_name,total_rows,backslash_n_count,backslash_n_percentage
0,movie_analytics.bronze.imdb_title_principals,job,101830563,82199797,80.7221
1,movie_analytics.bronze.imdb_title_principals,characters,101830563,52185276,51.2472
2,movie_analytics.bronze.imdb_name_basics,deathYear,15668421,15405054,98.3191
3,movie_analytics.bronze.imdb_name_basics,birthYear,15668421,14986500,95.6478
4,movie_analytics.bronze.imdb_title_basics,endYear,12801419,12639581,98.7358
5,movie_analytics.bronze.imdb_title_basics,runtimeMinutes,12801419,8004048,62.5247
6,movie_analytics.bronze.imdb_title_crew,writers,12797249,6323105,49.4099
7,movie_analytics.bronze.imdb_title_crew,directors,12797249,5685193,44.4251
8,movie_analytics.bronze.imdb_name_basics,primaryProfession,15668421,3177194,20.2777
9,movie_analytics.bronze.imdb_name_basics,knownForTitles,15668421,1893385,12.0841
